# Optimizacion de Rutas Escolares via Set Cover

Este notebook implementa un enfoque alternativo para el problema de rutas de buses escolares usando **programacion entera (ILP)** con la formulacion de **Set Cover**.

## Idea central

1. **Generamos** un pool de rutas candidatas: para cada estudiante como *centro*, construimos una ruta greedy con los `bus_capacity` vecinos mas cercanos segun distancias reales de la red vial.
2. **Formulamos** el Set Cover: seleccionar el **minimo numero de rutas** del pool tal que cada estudiante quede cubierto por al menos una ruta.
3. **Visualizamos** las rutas seleccionadas sobre el mapa real de Medellin.

### Formulacion matematica

Sea $J = \{1, \ldots, n\}$ el conjunto de estudiantes y $\mathcal{R} = \{r_1, \ldots, r_m\}$ el pool de rutas candidatas, donde cada $r_j \subseteq J$ con $|r_j| \leq C$.

**Variables:** $x_j \in \{0, 1\}$: 1 si la ruta $j$ es seleccionada.

$$\min \sum_{j=1}^{m} x_j$$

$$\text{s.t.} \quad \sum_{j \,:\, i \in r_j} x_j \geq 1 \quad \forall\, i \in J$$

$$x_j \in \{0, 1\}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import contextily as ctx
import pulp
from tqdm import tqdm

## 1. Carga de datos del escenario

Leemos `scenario_data.npz` generado por `data_generation.py`.

In [ ]:
data = np.load("scenario_data.npz")
dist_matrix = data["dist_matrix"]
x = data["x"]
y = data["y"]
origin_idx = int(data["origin_index"])
bus_capacity = int(data["bus_capacity"])

n_points = dist_matrix.shape[0]
children_idx = [i for i in range(n_points) if i != origin_idx]
n_children = len(children_idx)

D_children = dist_matrix[np.ix_(children_idx, children_idx)]

print(f"Total de estudiantes  : {n_children}")
print(f"Capacidad por bus     : {bus_capacity}")
print(f"Buses minimos teoricos: {int(np.ceil(n_children / bus_capacity))}")

## 2. Generacion de rutas candidatas

Para cada estudiante $i$ como centro, creamos una ruta greedy tomando sus `bus_capacity` vecinos mas cercanos. El resultado es un pool de $m \leq n$ rutas unicas, cada una con $|r_j| \leq C$.

In [ ]:
def build_candidate_route(center, D, capacity):
    dists = sorted((D[center, j], j) for j in range(D.shape[0]) if j != center)
    members = {center}
    for _, j in dists:
        if len(members) >= capacity:
            break
        members.add(j)
    return frozenset(members)


seen = set()
unique_routes = []

for i in tqdm(range(n_children), desc="Generando candidatos"):
    r = build_candidate_route(i, D_children, bus_capacity)
    if r not in seen:
        seen.add(r)
        unique_routes.append(r)

m = len(unique_routes)
print(f"Rutas candidatas unicas: {m}")

## 3. Formulacion y resolucion del Set Cover (ILP)

Usamos **PuLP** con el solver CBC para resolver la cobertura minima.

In [ ]:
coverage = {i: [] for i in range(n_children)}
for j, route in enumerate(unique_routes):
    for student in route:
        coverage[student].append(j)

prob = pulp.LpProblem("SetCover_BusRoutes", pulp.LpMinimize)
route_vars = [pulp.LpVariable(f"x_{j}", cat="Binary") for j in range(m)]

prob += pulp.lpSum(route_vars)

for i in range(n_children):
    covering = [route_vars[j] for j in coverage[i]]
    if covering:
        prob += pulp.lpSum(covering) >= 1

prob.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"Estado   : {pulp.LpStatus[prob.status]}")
print(f"Buses    : {int(pulp.value(prob.objective))}")

## 4. Analisis de la solucion

In [ ]:
selected_routes = [
    unique_routes[j]
    for j, var in enumerate(route_vars)
    if pulp.value(var) == 1
]
n_buses = len(selected_routes)
sizes = [len(r) for r in selected_routes]

print(f"Buses usados         : {n_buses}")
print(f"Tamano min.          : {min(sizes)} estudiantes")
print(f"Tamano max.          : {max(sizes)} estudiantes")
print(f"Tamano promedio      : {np.mean(sizes):.1f} estudiantes")

covered = set().union(*selected_routes)
print(f"Estudiantes cubiertos: {len(covered)} / {n_children}")


def nn_tour_distance(global_members, D_full, origin):
    remaining = list(global_members)
    current = origin
    total = 0.0
    while remaining:
        nearest = min(remaining, key=lambda n: D_full[current, n])
        total += D_full[current, nearest]
        remaining.remove(nearest)
        current = nearest
    total += D_full[current, origin]
    return total


route_distances = []
total_distance = 0.0
for route in selected_routes:
    members = [children_idx[i] for i in route]
    d = nn_tour_distance(members, dist_matrix, origin_idx)
    route_distances.append(d)
    total_distance += d

print(f"\nDistancia total de la flota: {total_distance/1000:.2f} km")
print(f"Distancia media por bus    : {np.mean(route_distances)/1000:.2f} km")

## 5. Visualizacion sobre el mapa de Medellin

In [ ]:
def distinct_colors(n):
    hues = np.linspace(0, 1, n, endpoint=False)
    return [mcolors.hsv_to_rgb((h, 0.75, 0.88)) for h in hues]


colors = distinct_colors(n_buses)
fig, ax = plt.subplots(figsize=(14, 14))

for idx, (route, color) in enumerate(zip(selected_routes, colors)):
    members = [children_idx[i] for i in route]
    ax.scatter([x[i] for i in members], [y[i] for i in members],
               color=color, s=20, zorder=3, label=f"Bus {idx+1} ({len(route)} est.)")

ax.scatter(x[origin_idx], y[origin_idx], marker="*",
           color="red", s=300, zorder=6, label="Colegio")

ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
ax.set_title(
    f"Set Cover --- {n_buses} buses, {total_distance/1000:.1f} km totales",
    fontsize=14,
)
ax.legend(fontsize=7, loc="upper left", ncol=3)

try:
    ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik,
                    zoom=14, crs="EPSG:4326")
except Exception:
    pass

fig.savefig("setcover_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Mapa guardado en setcover_map.png")

## 6. Guardar resultados

In [ ]:
max_len = max(len(r) for r in selected_routes)
routes_arr = np.full((n_buses, max_len), -1, dtype=np.int64)
for i, route in enumerate(selected_routes):
    members = [children_idx[j] for j in route]
    routes_arr[i, :len(members)] = members

np.savez(
    "setcover_result.npz",
    routes=routes_arr,
    n_buses=n_buses,
    total_distance_m=total_distance,
    route_distances_m=np.array(route_distances),
)
print("Resultados guardados en setcover_result.npz")